In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
ckpt_dir = "/content/drive/MyDrive/CommonCrawl/checkpoints_lab4"
if os.path.exists(ckpt_dir):
    files = os.listdir(ckpt_dir)
    print("Файлы в checkpoints_lab4:")
    for f in files:
        path = os.path.join(ckpt_dir, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  {f}  ({size_mb:.1f} MB)")
else:
    print("Папки нет!")

Файлы в checkpoints_lab4:
  gpt-epoch=00-val_perplexity=0.00.ckpt  (370.7 MB)
  gpt-epoch=00-val_perplexity=0.00-v1.ckpt  (370.7 MB)
  gpt-epoch=01-val_perplexity=0.00.ckpt  (370.7 MB)
  last.ckpt  (370.7 MB)


In [3]:
%cd /content
import os
if not os.path.exists('/content/MNNA-2026'):
    !git clone https://github.com/ksenkap/MNNA-2026.git
else:
    print("Репозиторий уже склонирован.")
%cd /content/MNNA-2026
!git fetch --all
!git checkout lab4
!git pull origin lab4
!git branch

/content
Cloning into 'MNNA-2026'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 134 (delta 55), reused 11 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 118.06 KiB | 5.62 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/MNNA-2026
branch 'lab4' set up to track 'origin/lab4'.
Switched to a new branch 'lab4'
From https://github.com/ksenkap/MNNA-2026
 * branch            lab4       -> FETCH_HEAD
Already up to date.
* lab4
  main


In [ ]:
import os
print("Файлы в src/models/:", os.listdir("src/models/"))
print("Файлы в configs/:", os.listdir("configs/"))
print("Файлы в notebooks/:", os.listdir("notebooks/"))

Файлы в src/models/: ['.gitkeep', 'gpt.py', 'ffn.py', 'attention.py', '__init__.py', 'lightning_module.py', 'transformer_layer.py', 'positional_encoding.py']
Файлы в configs/: ['.gitkeep', 'gpt_config.yaml']
Файлы в notebooks/: ['.gitkeep']


In [ ]:
from datasets import load_dataset
import time

t0 = time.time()
ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
print(f"Загружено за {time.time()-t0:.1f} сек")
print(f"Train строк: {len(ds['train'])}")
print(f"Validation строк: {len(ds['validation'])}")
print(f"Test строк: {len(ds['test'])}")

# Смотрим на первую непустую строку
for i in range(100):
    if ds['train'][i]['text'].strip():
        print(f"\nПример строки [{i}]:")
        print(ds['train'][i]['text'][:300])
        break

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Загружено за 10.8 сек
Train строк: 1801350
Validation строк: 3760
Test строк: 4358

Пример строки [1]:
 = Valkyria Chronicles III = 



In [ ]:
N_LINES = 300_000   # <-- было 50 000, теперь 300 000

train_lines = ds['train']['text'][:N_LINES]
print(f"Взято строк: {len(train_lines)}")

chunks = []
current_chunk = []

for line in train_lines:
    line = line.strip()
    if line == "":
        if current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
    else:
        if not line.startswith("="):   # пропускаем заголовки
            current_chunk.append(line)

if current_chunk:
    chunks.append(" ".join(current_chunk))

print(f"Получено объектов (абзацев): {len(chunks)}")

# Оценка объёма
total_chars = sum(len(c) for c in chunks)
print(f"Всего символов: {total_chars:,}")
print(f"Примерный размер: {total_chars / 1e6:.1f} MB текста")

Взято строк: 300000
Получено объектов (абзацев): 44605
Всего символов: 87,191,967
Примерный размер: 87.2 MB текста


In [ ]:
import os
from tokenizers import Tokenizer
import numpy as np

# Загружаем токенизатор из ЛР1
tokenizer_path = "/content/drive/MyDrive/CommonCrawl/bpe_tokenizer.json"
assert os.path.exists(tokenizer_path), f"Нет токенизатора: {tokenizer_path}"
tokenizer = Tokenizer.from_file(tokenizer_path)
print(f"Токенизатор загружен, словарь: {tokenizer.get_vocab_size()}")

# Токенизируем все чанки
import time
t0 = time.time()
all_ids = []
for c in chunks:
    enc = tokenizer.encode(c)
    all_ids.append(enc.ids)
print(f"Токенизация за {time.time()-t0:.1f} сек")
print(f"Всего объектов: {len(all_ids)}")
print(f"Всего токенов: {sum(len(ids) for ids in all_ids):,}")

Токенизатор загружен, словарь: 10000
Токенизация за 38.7 сек
Всего объектов: 44605
Всего токенов: 24,631,732


In [ ]:
def packed_batching(all_token_ids, batch_size=512, pad_token_id=0):
    packed_batches = []
    attention_masks = []
    current_batch = []
    current_mask = []
    object_counter = 1

    for token_ids in all_token_ids:
        i = 0
        while i < len(token_ids):
            space_left = batch_size - len(current_batch)
            tokens_to_add = token_ids[i:i + space_left]
            current_batch.extend(tokens_to_add)
            current_mask.extend([object_counter] * len(tokens_to_add))
            i += space_left
            object_counter += 1
            if len(current_batch) == batch_size:
                packed_batches.append(current_batch)
                attention_masks.append(current_mask)
                current_batch = []
                current_mask = []
                object_counter = 1

    if current_batch:
        pad_count = batch_size - len(current_batch)
        current_batch.extend([pad_token_id] * pad_count)
        current_mask.extend([0] * pad_count)
        packed_batches.append(current_batch)
        attention_masks.append(current_mask)

    return packed_batches, attention_masks


BATCH_SIZE = 512
print(f"Packed batching (BATCH_SIZE={BATCH_SIZE})...")
packed_batches, attention_masks = packed_batching(all_ids, batch_size=BATCH_SIZE)
print(f"Батчей: {len(packed_batches)}")
print(f"Общая длина: {sum(len(b) for b in packed_batches):,}")

# Сохраняем на Drive (НЕ перезаписываем старый файл!)
save_path = "/content/drive/MyDrive/CommonCrawl/wikitext_packed_batches_large.npz"
np.savez_compressed(
    save_path,
    batches=np.array(packed_batches, dtype=np.int32),
    masks=np.array(attention_masks, dtype=np.int8),
    batch_size=BATCH_SIZE,
)
print(f"Сохранено: {save_path}")
print(f"Размер: {os.path.getsize(save_path) / 1024**2:.1f} MB")

Packed batching (BATCH_SIZE=512)...
Батчей: 48109
Общая длина: 24,631,808
Сохранено: /content/drive/MyDrive/CommonCrawl/wikitext_packed_batches_large.npz
Размер: 31.9 MB


In [ ]:
%cd /content/MNNA-2026
!git checkout lab4
!git pull origin lab4
!cat configs/gpt_config.yaml

/content/MNNA-2026
Already on 'lab4'
Your branch is up to date with 'origin/lab4'.
From https://github.com/ksenkap/MNNA-2026
 * branch            lab4       -> FETCH_HEAD
Already up to date.
model:
  vocab_size: 10000
  d_model: 256
  n_heads: 8
  n_layers: 6
  d_ff: 1024
  max_seq_len: 512
  dropout: 0.15

training:
  batch_size: 32
  learning_rate: 3e-4
  weight_decay: 0.01
  max_epochs: 40
  warmup_steps: 1000
  max_grad_norm: 1.0
  checkpoint_dir: checkpoints/
  log_dir: logs/


In [ ]:
%%writefile /content/MNNA-2026/configs/gpt_config.yaml
model:
  vocab_size: 10000
  d_model: 512
  n_heads: 8
  n_kv_heads: 2         # GQA: 8 Q-голов / 2 K/V-группы = по 4 Q на группу
  n_layers: 8
  d_ff: 2048
  max_seq_len: 512
  dropout: 0.15         # против переобучения (было 0.1)

training:
  batch_size: 32
  learning_rate: 3e-4
  weight_decay: 0.05    # против переобучения (было 0.01)
  max_epochs: 10        # меньше эпох, т.к. датасет в 8.5 раз больше и есть риск переобучения
  warmup_steps: 2000
  max_grad_norm: 1.0
  checkpoint_dir: /content/drive/MyDrive/CommonCrawl/checkpoints_lab4/
  log_dir: /content/drive/MyDrive/CommonCrawl/logs_lab4/

Overwriting /content/MNNA-2026/configs/gpt_config.yaml


In [14]:
%%writefile /content/MNNA-2026/src/training/train.py
import os
from dotenv import load_dotenv
from omegaconf import OmegaConf

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from src.models.lightning_module import GPTLightningModule
from src.data.wikitext_datamodule import WikiTextDataModule


def main():
    # ---------------------------------------------------------------
    # 1. Переменные окружения
    # ---------------------------------------------------------------
    # .env может не быть — это не критично для ЛР4
    load_dotenv("/content/MNNA-2026/.env", override=False)

    project_name = os.getenv("CLEARML_PROJECT_NAME", "MNNA-2026")

    # ---------------------------------------------------------------
    # 2. Конфиг
    # ---------------------------------------------------------------
    cfg = OmegaConf.load("/content/MNNA-2026/configs/gpt_config.yaml")
    print("Конфиг:")
    print(OmegaConf.to_yaml(cfg))

    # Пути берём из конфига
    checkpoint_dir = cfg.training.checkpoint_dir
    log_dir = cfg.training.log_dir
    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    print(f"Чекпоинты: {checkpoint_dir}")
    print(f"Логи:      {log_dir}")

    # ---------------------------------------------------------------
    # 3. DataModule — используем НОВЫЙ датасет из ЛР4
    # ---------------------------------------------------------------
    data_dir = "/content/drive/MyDrive/CommonCrawl"
    npz_path = os.path.join(data_dir, "wikitext_packed_batches_large.npz")
    assert os.path.exists(npz_path), f"Нет датасета: {npz_path}"

    dm = WikiTextDataModule(
        npz_path=npz_path,
        batch_size=cfg.training.batch_size,
        val_split=0.05,
    )

    # ---------------------------------------------------------------
    # 4. Модель (GPT + GQA внутри attention)
    # ---------------------------------------------------------------
    model = GPTLightningModule(cfg)

    # ---------------------------------------------------------------
    # 5. Callbacks
    # ---------------------------------------------------------------
    checkpoint_callback = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename="gpt-{epoch:02d}-{val_perplexity:.2f}",
        monitor="val/perplexity",
        mode="min",
        save_top_k=3,
        save_last=True,           # сохраняем last.ckpt для resume
        verbose=True,
    )

    lr_monitor = LearningRateMonitor(logging_interval="step")

    # ---------------------------------------------------------------
    # 6. Логгер (TensorBoard)
    # ---------------------------------------------------------------
    tb_logger = TensorBoardLogger(
        save_dir=log_dir,
        name="gpt",
    )

    # ---------------------------------------------------------------
    # 7. Опционально: ClearML — если есть .env с ключами
    # ---------------------------------------------------------------
    clearml_logger = None
    try:
        from clearml import Task
        access_key = os.getenv("CLEARML_API_ACCESS_KEY")
        secret_key = os.getenv("CLEARML_API_SECRET_KEY")
        if access_key and secret_key:
            task = Task.init(
                project_name=project_name,
                task_name="Lab4 training",
                auto_connect_frameworks=False,
            )
            print(f"ClearML подключён, task id: {task.id}")
        else:
            print("ClearML не подключён (нет ключей в .env) — это ок")
    except Exception as e:
        print(f"ClearML недоступен: {e}")

    # ---------------------------------------------------------------
    # 8. Trainer
    # ---------------------------------------------------------------
    trainer = pl.Trainer(
        max_epochs=cfg.training.max_epochs,
        accelerator="gpu",
        devices=1,
        gradient_clip_val=cfg.training.max_grad_norm,
        gradient_clip_algorithm="norm",
        callbacks=[checkpoint_callback, lr_monitor],
        logger=tb_logger,
        log_every_n_steps=50,
        val_check_interval=0.5,     # проверка валидации дважды за эпоху
        precision="16-mixed",       # fp16 на T4
    )

    # ---------------------------------------------------------------
    # 9. Resume из last.ckpt, если есть
    # ---------------------------------------------------------------
    last_ckpt = os.path.join(checkpoint_dir, "last.ckpt")
    resume_path = last_ckpt if os.path.exists(last_ckpt) else None
    if resume_path:
        print(f"Продолжаем обучение с: {resume_path}")
    else:
        print("Начинаем обучение с нуля.")

    # ---------------------------------------------------------------
    # 10. Запуск
    # ---------------------------------------------------------------
    trainer.fit(model, datamodule=dm, ckpt_path=resume_path)

    print("\n=== Обучение завершено ===")
    print("Лучший чекпоинт:", checkpoint_callback.best_model_path)
    print("Лучшая perplexity:", checkpoint_callback.best_model_score)


if __name__ == "__main__":
    main()

Overwriting /content/MNNA-2026/src/training/train.py


In [15]:
%%writefile /content/MNNA-2026/src/models/attention.py
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    """
    Многоголовое внимание с поддержкой:
      - packed batching (seq_ids: 0=PAD, 1,2,...=номера объектов),
      - GQA (Grouped Query Attention): n_kv_heads <= n_heads.

    При n_kv_heads == n_heads — обычный MHA.
    При n_kv_heads < n_heads  — GQA: несколько Q-голов делят одну пару K/V.
    При n_kv_heads == 1       — MQA (Multi-Query Attention).

    seq_ids: (batch, seq_len)
    Внимание между i и j разрешено, только если:
        - seq_ids[i] == seq_ids[j] (один объект),
        - j <= i (causal),
        - seq_ids[i] != 0 (не PAD).
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        # По умолчанию n_kv_heads == n_heads (обычный MHA)
        if n_kv_heads is None:
            n_kv_heads = n_heads

        assert n_heads % n_kv_heads == 0, \
            f"n_heads ({n_heads}) must be divisible by n_kv_heads ({n_kv_heads})"

        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads      # сколько Q-голов на одну K/V
        self.d_head = d_model // n_heads

        # Проекции Q — полные (n_heads), K/V — сжатые (n_kv_heads)
        self.W_q = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        self.dropout = nn.Dropout(p=dropout)

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, n_kv_heads, seq_len, d_head)
        Возвращает: (batch, n_heads, seq_len, d_head),
        где каждая K/V-голова повторяется n_rep раз.
        """
        if self.n_rep == 1:
            return x
        return x.repeat_interleave(self.n_rep, dim=1)

    def forward(self, x: torch.Tensor, seq_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        device = x.device

        # 1. Проекции
        q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        k = self.W_k(x).view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        # q: (B, n_heads,    T, d_head)
        # k: (B, n_kv_heads, T, d_head)
        # v: (B, n_kv_heads, T, d_head)

        # 2. Расширяем K/V до n_heads (GQA: общие K/V для группы Q-голов)
        k = self._repeat_kv(k)   # (B, n_heads, T, d_head)
        v = self._repeat_kv(v)   # (B, n_heads, T, d_head)

        # 3. Оценки внимания
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        # (B, n_heads, T, T)

        # 4. Маска
        s_i = seq_ids.unsqueeze(1).unsqueeze(-1)   # (B, 1, T, 1)
        s_j = seq_ids.unsqueeze(1).unsqueeze(-2)   # (B, 1, 1, T)

        same_object = (s_i == s_j)
        i_idx = torch.arange(seq_len, device=device).view(1, 1, seq_len, 1)
        j_idx = torch.arange(seq_len, device=device).view(1, 1, 1, seq_len)
        causal = (j_idx <= i_idx)
        not_pad = (s_i != 0)

        mask = same_object & causal & not_pad
        scores = scores.masked_fill(~mask, float("-inf"))

        # 5. Softmax
        attn = torch.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        # 6. Взвешенная сумма
        out = torch.matmul(attn, v)   # (B, n_heads, T, d_head)

        # 7. Склейка голов
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # 8. Финальная проекция
        out = self.W_o(out)
        return out

Overwriting /content/MNNA-2026/src/models/attention.py


In [16]:
%%writefile /content/MNNA-2026/src/models/transformer_layer.py
import torch
import torch.nn as nn

from src.models.attention import MultiHeadAttention
from src.models.ffn import FFN


class TransformerLayer(nn.Module):
    """
    Один слой трансформера в post-norm варианте:

        z1 = LayerNorm(x + Attention(x))
        z2 = LayerNorm(z1 + FFN(z1))

    post-norm — нормализация применяется ПОСЛЕ residual connection.
    Поддерживает GQA через n_kv_heads.
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        n_kv_heads: int = None,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()

        self.attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            dropout=dropout,
        )
        self.ffn = FFN(d_model=d_model, d_ff=d_ff, dropout=dropout, activation=activation)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor, seq_ids: torch.Tensor) -> torch.Tensor:
        # Подслой внимания с residual + post-norm
        attn_out = self.attn(x, seq_ids)
        x = self.norm1(x + self.dropout(attn_out))

        # Подслой FFN с residual + post-norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

Overwriting /content/MNNA-2026/src/models/transformer_layer.py


In [17]:
%%writefile /content/MNNA-2026/src/models/gpt.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models.positional_encoding import SinusoidalPositionalEncoding
from src.models.transformer_layer import TransformerLayer


class GPT(nn.Module):
    """
    GPT-like модель с GQA и KV-кэшем.
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        n_kv_heads: int = None,
        max_seq_len: int = 512,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads if n_kv_heads is not None else n_heads
        self.n_layers = n_layers

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.pos_encoding = SinusoidalPositionalEncoding(
            d_model=d_model, max_seq_len=max_seq_len, dropout=dropout
        )

        self.layers = nn.ModuleList([
            TransformerLayer(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=self.n_kv_heads,
                d_ff=d_ff,
                dropout=dropout,
                activation=activation,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(
        self,
        input_ids: torch.Tensor,
        seq_ids: torch.Tensor,
        past_kvs=None,
        position_offset: int = 0,
        use_cache: bool = False,
    ):
        """
        input_ids: (B, T_new)
        seq_ids: (B, T_new)
        past_kvs: list[(k, v)] * n_layers, или None
        position_offset: int — сдвиг позиций (для инференса с KV-кэшем)
        use_cache: bool — если True, возвращает new_past_kvs (для инференса)

        Возвращает:
            logits: (B, T_new, vocab_size)
            new_past_kvs: list[(k, v)] * n_layers, или None
        """
        x = self.token_embedding(input_ids)
        x = self.pos_encoding(x, seq_ids, position_offset=position_offset)

        new_past_kvs = [] if use_cache else None

        for i, layer in enumerate(self.layers):
            past_kv = past_kvs[i] if past_kvs is not None else None
            x, new_kv = layer(x, seq_ids, past_kv=past_kv)
            if use_cache:
                new_past_kvs.append(new_kv)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, new_past_kvs

    def compute_loss(self, logits, targets, seq_ids):
        shift_logits = logits[:, :-1, :].contiguous()
        shift_targets = targets[:, 1:].contiguous()

        s_i = seq_ids[:, :-1]
        s_next = seq_ids[:, 1:]
        loss_mask = (s_i == s_next) & (s_i != 0)

        loss_per_token = F.cross_entropy(
            shift_logits.view(-1, self.vocab_size),
            shift_targets.view(-1),
            reduction="none",
        )
        loss_per_token = loss_per_token.view(shift_targets.size(0), -1)

        loss_mask = loss_mask.float()
        loss = (loss_per_token * loss_mask).sum() / loss_mask.sum().clamp(min=1.0)
        return loss

Overwriting /content/MNNA-2026/src/models/gpt.py


In [18]:
%%writefile /content/MNNA-2026/src/models/lightning_module.py
import math
import torch
import pytorch_lightning as pl

from src.models.gpt import GPT


class GPTLightningModule(pl.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.save_hyperparameters(ignore=["cfg"])

        n_kv_heads = cfg.model.get("n_kv_heads", None)

        self.model = GPT(
            vocab_size=cfg.model.vocab_size,
            d_model=cfg.model.d_model,
            n_heads=cfg.model.n_heads,
            n_layers=cfg.model.n_layers,
            d_ff=cfg.model.d_ff,
            n_kv_heads=n_kv_heads,
            max_seq_len=cfg.model.max_seq_len,
            dropout=cfg.model.dropout,
        )

    def forward(self, input_ids, seq_ids):
        logits, _ = self.model(input_ids, seq_ids, use_cache=False)
        return logits

    def _shared_step(self, batch, stage: str):
        input_ids, seq_ids = batch
        logits = self(input_ids, seq_ids)
        loss = self.model.compute_loss(logits, input_ids, seq_ids)
        perplexity = torch.exp(loss)

        self.log(f"{stage}/loss", loss, prog_bar=True,
                 on_step=(stage == "train"), on_epoch=True)
        self.log(f"{stage}/perplexity", perplexity, prog_bar=True,
                 on_step=False, on_epoch=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, stage="train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, stage="val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.cfg.training.learning_rate,
            weight_decay=self.cfg.training.weight_decay,
        )

        warmup_steps = self.cfg.training.warmup_steps

        def lr_lambda(current_step):
            if current_step < warmup_steps:
                return float(current_step) / float(max(1, warmup_steps))
            progress = (current_step - warmup_steps) / max(1, 200_000)
            return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lr_lambda=lr_lambda
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
                "name": "lr",
            },
        }

    def on_after_backward(self):
        total_norm = 0.0
        for p in self.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        self.log("train/grad_norm", total_norm, on_step=True, on_epoch=False)

Overwriting /content/MNNA-2026/src/models/lightning_module.py


In [ ]:
!pip install -q pytorch-lightning omegaconf clearml tensorboard tokenizers datasets python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 49.6 MB/s eta 0:00:00


In [ ]:
%cd /content/MNNA-2026
!echo "=== attention.py (только forward) ==="
!grep -n "def forward" src/models/attention.py
!echo "=== gpt.py ==="
!cat src/models/gpt.py | head -100
!echo "=== positional_encoding.py ==="
!cat src/models/positional_encoding.py

In [19]:
%%writefile /content/MNNA-2026/src/models/positional_encoding.py
import torch
import torch.nn as nn
import math


class SinusoidalPositionalEncoding(nn.Module):
    """
    Синусоидальное позиционное кодирование с поддержкой:
      - packed batching (seq_ids: 0=PAD, 1,2,...=номера объектов),
      - инференса с KV-кэшем (position_offset — сдвиг позиций).
    """

    def __init__(self, d_model: int, max_seq_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_seq_len, d_model)
        self.register_buffer("pe", pe)

    def forward(
        self,
        x: torch.Tensor,
        seq_ids: torch.Tensor,
        position_offset: int = 0,
    ) -> torch.Tensor:
        """
        x: (batch, seq_len, d_model)
        seq_ids: (batch, seq_len)
        position_offset: int — сдвиг позиций (для инференса с KV-кэшем).
            Например, если в кэше уже 10 токенов, а мы обрабатываем
            новый токен (seq_len=1), то position_offset=10.
        """
        batch_size, seq_len, _ = x.shape
        device = x.device

        positions = torch.zeros(batch_size, seq_len, dtype=torch.long, device=device)

        for b in range(batch_size):
            pos = position_offset
            for i in range(seq_len):
                sid = seq_ids[b, i].item()
                if sid == 0:
                    positions[b, i] = 0
                else:
                    if i > 0 and seq_ids[b, i - 1].item() == sid:
                        pos += 1
                    else:
                        pos = position_offset
                    positions[b, i] = pos

        pe = self.pe[0, positions]  # (batch, seq_len, d_model)
        return self.dropout(x + pe)

Overwriting /content/MNNA-2026/src/models/positional_encoding.py


In [20]:
%%writefile /content/MNNA-2026/src/models/attention.py
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    """
    Многоголовое внимание с поддержкой:
      - packed batching (seq_ids),
      - GQA (n_kv_heads <= n_heads),
      - KV-кэш для инференса (past_kv).

    forward возвращает (out, new_kv), где new_kv = (k, v) — обновлённый кэш.
    Если past_kv=None — это первый вызов, кэш создаётся с нуля.
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        if n_kv_heads is None:
            n_kv_heads = n_heads

        assert n_heads % n_kv_heads == 0, \
            f"n_heads ({n_heads}) must be divisible by n_kv_heads ({n_kv_heads})"

        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads

        self.W_q = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        self.dropout = nn.Dropout(p=dropout)

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, n_kv_heads, T, d_head) → (B, n_heads, T, d_head)."""
        if self.n_rep == 1:
            return x
        return x.repeat_interleave(self.n_rep, dim=1)

    def forward(
        self,
        x: torch.Tensor,
        seq_ids: torch.Tensor,
        past_kv=None,
    ):
        """
        x: (B, T_new, d_model) — новые токены (при инференсе T_new=1).
        seq_ids: (B, T_new) — их seq_ids.
        past_kv: (past_k, past_v) или None.
            past_k: (B, n_kv_heads, T_past, d_head)
            past_v: (B, n_kv_heads, T_past, d_head)

        Возвращает (out, new_kv):
            out: (B, T_new, d_model)
            new_kv: (k_all, v_all), где k_all: (B, n_kv_heads, T_past+T_new, d_head)
        """
        batch_size, seq_len, _ = x.shape
        device = x.device

        # 1. Проекции (только для новых токенов!)
        q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        k_new = self.W_k(x).view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)
        v_new = self.W_v(x).view(batch_size, seq_len, self.n_kv_heads, self.d_head).transpose(1, 2)

        # 2. Приклеиваем новые K/V к кэшу
        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k_new], dim=2)   # (B, n_kv_heads, T_past+T_new, d_head)
            v = torch.cat([past_v, v_new], dim=2)
        else:
            k = k_new
            v = v_new

        new_kv = (k, v)

        # 3. Расширяем K/V до n_heads (GQA)
        k_rep = self._repeat_kv(k)   # (B, n_heads, T_total, d_head)
        v_rep = self._repeat_kv(v)

        # 4. Оценки внимания
        # q: (B, n_heads, T_new, d_head), k_rep: (B, n_heads, T_total, d_head)
        scores = torch.matmul(q, k_rep.transpose(-2, -1)) / math.sqrt(self.d_head)
        # scores: (B, n_heads, T_new, T_total)

        # 5. Маска
        T_new = seq_len
        T_total = k.size(2)
        T_past = T_total - T_new

        # Позиции новых токенов в полной последовательности
        i_idx = T_past + torch.arange(T_new, device=device)   # (T_new,)
        j_idx = torch.arange(T_total, device=device)          # (T_total,)

        # Causal: j <= i
        causal = (j_idx.unsqueeze(0) <= i_idx.unsqueeze(1))    # (T_new, T_total)

        # Маска по seq_ids: только для режима обучения (packed batching)
        # При инференсе seq_ids = все единицы → маска по seq_ids не нужна
        if past_kv is None:
            # Packed batching: работаем как раньше
            s_i = seq_ids.unsqueeze(1).unsqueeze(-1)   # (B, 1, T_new, 1)
            s_j = seq_ids.unsqueeze(1).unsqueeze(-2)   # (B, 1, 1, T_new)
            same_object = (s_i == s_j)                 # (B, 1, T_new, T_new)
            not_pad = (s_i != 0)
            mask = same_object & causal.unsqueeze(0) & not_pad
        else:
            # Инференс: считаем, что все seq_ids одинаковые
            mask = causal.unsqueeze(0).unsqueeze(0).expand(batch_size, self.n_heads, T_new, T_total)

        scores = scores.masked_fill(~mask, float("-inf"))

        # 6. Softmax
        attn = torch.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.dropout(attn)

        # 7. Взвешенная сумма
        out = torch.matmul(attn, v_rep)   # (B, n_heads, T_new, d_head)

        # 8. Склейка голов
        out = out.transpose(1, 2).contiguous().view(batch_size, T_new, self.d_model)

        # 9. Финальная проекция
        out = self.W_o(out)
        return out, new_kv

Overwriting /content/MNNA-2026/src/models/attention.py


In [21]:
%%writefile /content/MNNA-2026/src/models/transformer_layer.py
import torch
import torch.nn as nn

from src.models.attention import MultiHeadAttention
from src.models.ffn import FFN


class TransformerLayer(nn.Module):
    """
    Один слой трансформера (post-norm), поддерживает KV-кэш.
    Возвращает (out, new_kv).
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        n_kv_heads: int = None,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()

        self.attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            dropout=dropout,
        )
        self.ffn = FFN(d_model=d_model, d_ff=d_ff, dropout=dropout, activation=activation)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, seq_ids, past_kv=None):
        attn_out, new_kv = self.attn(x, seq_ids, past_kv=past_kv)
        x = self.norm1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x, new_kv

Overwriting /content/MNNA-2026/src/models/transformer_layer.py


In [ ]:
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')
import torch

import src.models.attention as attn_mod
import src.models.transformer_layer as tl_mod
import src.models.gpt as gpt_mod
import src.models.lightning_module as lm_mod
import src.models.positional_encoding as pe_mod

for m in [pe_mod, attn_mod, tl_mod, gpt_mod, lm_mod]:
    importlib.reload(m)

from src.models.gpt import GPT

torch.manual_seed(42)
device = 'cuda'

model = GPT(
    vocab_size=1000,
    d_model=128,
    n_heads=8,
    n_kv_heads=2,
    n_layers=2,
    d_ff=512,
    max_seq_len=64,
    dropout=0.0,
).to(device).eval()

# ============================================================
# Тест 1: обычный forward на всю последовательность
# ============================================================
B, N = 2, 16
input_ids = torch.randint(0, 1000, (B, N), device=device)
seq_ids = torch.ones(B, N, dtype=torch.long, device=device)  # все токены — одна последовательность

with torch.no_grad():
    logits_full, _ = model(input_ids, seq_ids, use_cache=False)
print("Full forward logits shape:", logits_full.shape)

# ============================================================
# Тест 2: пошаговый forward с KV-кэшем
# ============================================================
past_kvs = None
logits_cached = []
for t in range(N):
    tok = input_ids[:, t:t+1]        # (B, 1)
    sid = seq_ids[:, t:t+1]          # (B, 1)
    with torch.no_grad():
        logit_t, past_kvs = model(
            tok, sid,
            past_kvs=past_kvs,
            position_offset=t,
            use_cache=True,
        )
    logits_cached.append(logit_t)

logits_cached = torch.cat(logits_cached, dim=1)  # (B, N, vocab)
print("Cached forward logits shape:", logits_cached.shape)

# Сравнение
diff = (logits_full - logits_cached).abs().max().item()
print(f"Max abs diff (full vs cached): {diff:.2e}")

if diff < 1e-3:
    print("✅ KV-кэш даёт идентичный результат")
else:
    print("❌ Расхождение выше нормы")

# ============================================================
# Тест 3: форма кэша
# ============================================================
print("\n=== Форма KV-кэша по слоям ===")
for i, (k, v) in enumerate(past_kvs):
    print(f"Layer {i}: k shape = {k.shape}, v shape = {v.shape}")
print(f"Ожидаемая форма: (B={B}, n_kv_heads={model.n_kv_heads}, T={N}, d_head={model.d_model // model.n_heads})")

# ============================================================
# Тест 4: сравнение памяти KV-кэша для GQA vs MHA
# ============================================================
d_head = model.d_model // model.n_heads

bytes_per_token_gqa = B * model.n_kv_heads * d_head * 2 * 2   # K и V, fp16
bytes_per_token_mha = B * model.n_heads    * d_head * 2 * 2

print(f"\nKV-кэш на 1 токен (fp16, batch={B}):")
print(f"  GQA (n_kv_heads={model.n_kv_heads}): {bytes_per_token_gqa} байт")
print(f"  MHA (n_heads={model.n_heads}):     {bytes_per_token_mha} байт")
print(f"  Экономия: {bytes_per_token_mha / bytes_per_token_gqa:.1f}x")

# ============================================================
# Тест 5: генерация с кэшем (итеративно предсказываем следующий токен)
# ============================================================
print("\n=== Тест генерации с KV-кэшем ===")
prompt = input_ids[:, :4]     # первые 4 токена как prompt
past_kvs = None
generated = prompt.clone()

for step in range(4):   # сгенерируем 4 токена
    # На первом шаге подаём весь prompt, дальше — по одному токену
    if step == 0:
        cur_tokens = prompt
        cur_seq_ids = torch.ones_like(prompt)
        pos_offset = 0
    else:
        cur_tokens = generated[:, -1:]     # последний токен
        cur_seq_ids = torch.ones_like(cur_tokens)
        pos_offset = generated.shape[1] - 1

    with torch.no_grad():
        logits, past_kvs = model(
            cur_tokens, cur_seq_ids,
            past_kvs=past_kvs,
            position_offset=pos_offset,
            use_cache=True,
        )
    next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # (B, 1)
    generated = torch.cat([generated, next_tok], dim=1)

print(f"Prompt shape:    {prompt.shape}")
print(f"Generated shape: {generated.shape}")
print("Сгенерированные токены (batch=0):", generated[0].tolist())
print("Сгенерированные токены (batch=1):", generated[1].tolist())
print("✅ Генерация с KV-кэшем работает.")

In [ ]:
!mkdir -p /content/drive/MyDrive/CommonCrawl/checkpoints_lab4
!mkdir -p /content/drive/MyDrive/CommonCrawl/logs_lab4
!ls -la /content/drive/MyDrive/CommonCrawl/

total 37191
drwx------ 6 root root     4096 Sep 18 19:00 .
drwx------ 4 root root     4096 Sep 18 18:39 ..
-rw------- 1 root root   638171 Sep 18 11:37 bpe_tokenizer.json
drwx------ 2 root root     4096 Sep 18 11:52 checkpoints
drwx------ 2 root root     4096 Sep 18 19:00 checkpoints_lab4
drwx------ 2 root root     4096 Sep 18 11:52 logs
drwx------ 2 root root     4096 Sep 18 19:00 logs_lab4
-rw------- 1 root root 33422469 Sep 18 18:44 wikitext_packed_batches_large.npz
-rw------- 1 root root  3997152 Sep 18 11:37 wikitext_packed_batches.npz


In [ ]:
import sys, importlib
sys.path.insert(0, '/content/MNNA-2026')
import torch
from omegaconf import OmegaConf

import src.models.attention as m1
import src.models.transformer_layer as m2
import src.models.gpt as m3
import src.models.lightning_module as m4
import src.data.wikitext_datamodule as m5
for m in [m1, m2, m3, m4, m5]:
    importlib.reload(m)

from src.data.wikitext_datamodule import WikiTextDataModule
from src.models.lightning_module import GPTLightningModule

cfg = OmegaConf.load('/content/MNNA-2026/configs/gpt_config.yaml')
print("Config:")
print(OmegaConf.to_yaml(cfg))

# DataModule
dm = WikiTextDataModule(
    npz_path="/content/drive/MyDrive/CommonCrawl/wikitext_packed_batches_large.npz",
    batch_size=cfg.training.batch_size,
    val_split=0.05,
)
dm.setup()
print(f"Train batches: {len(dm.train_dataset)}")
print(f"Val batches:   {len(dm.val_dataset)}")

# Модель
model = GPTLightningModule(cfg).cuda()
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

# Один forward+backward
batch = next(iter(dm.train_dataloader()))
input_ids, seq_ids = batch
input_ids = input_ids.cuda()
seq_ids = seq_ids.cuda()

print(f"input_ids.shape = {input_ids.shape}")
print(f"seq_ids.shape   = {seq_ids.shape}")

logits = model(input_ids, seq_ids)
loss = model.model.compute_loss(logits, input_ids, seq_ids)
loss.backward()

print(f"logits.shape = {logits.shape}")
print(f"loss = {loss.item():.4f}")
print(f"perplexity = {torch.exp(loss).item():.2f}")

# Проверка памяти
print(f"GPU memory: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB")

print("\n✅ Smoke-тест пройден, модель готова к обучению.")

In [ ]:
%cd /content/MNNA-2026
!sed -i 's/^  batch_size: 12/  batch_size: 16/' configs/gpt_config.yaml
!grep "batch_size" configs/gpt_config.yaml

/content/MNNA-2026
  batch_size: 16


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"Free: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

Free: 12.11 GB


In [ ]:
%cd /content/MNNA-2026
!python -m src.training.train

Выходные данные были обрезаны до нескольких последних строк (5000).
                                                               train/loss_epoch:
                                                               5.411            
                                                               train/perplexity:
Epoch 1/9  ━━━━━━━━╸━━━━━━━ 1584/2857 0:21:58 •       1.31it/s v_num: 1.000     
                                      0:16:14                  train/loss_step: 
                                                               4.166 val/loss:  
                                                               4.006            
                                                               val/perplexity:  
                                                               55.249           
                                                               train/loss_epoch:
                                                               5.411            
                                         

In [6]:
import os
import re

ckpt_dir = "/content/drive/MyDrive/CommonCrawl/checkpoints_lab4"
if os.path.exists(ckpt_dir):
    files = os.listdir(ckpt_dir)
    print("Файлы в checkpoints_lab4:")

    # Отфильтруем файлы .ckpt
    ckpt_files = [f for f in files if f.endswith('.ckpt')]

    # Функция для извлечения perplexity из имени файла
    def extract_perplexity(filename):
        # Ищем шаблон val_perplexity=ЧИСЛО, где ЧИСЛО - это цифры и точка
        # Используем более строгое выражение: \d+\.\d+ или \d+
        match = re.search(r'val_perplexity=(\d+\.\d+|\d+)', filename)
        if match:
            return float(match.group(1))
        return 999.0  # Для файлов без perplexity (например, last.ckpt)

    # Сортируем по perplexity
    ckpt_files.sort(key=extract_perplexity)

    for f in ckpt_files:
        path = os.path.join(ckpt_dir, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  {f}  ({size_mb:.1f} MB)")
else:
    print("Папки нет!")

Файлы в checkpoints_lab4:
  gpt-epoch=00-val_perplexity=0.00.ckpt  (370.7 MB)
  gpt-epoch=00-val_perplexity=0.00-v1.ckpt  (370.7 MB)
  gpt-epoch=01-val_perplexity=0.00.ckpt  (370.7 MB)
  last.ckpt  (370.7 MB)


In [8]:
!pip install -q pytorch-lightning omegaconf clearml tensorboard tokenizers datasets python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.9 MB/s eta 0:00:00


In [23]:
%%writefile /content/MNNA-2026/configs/gpt_config.yaml
model:
  vocab_size: 10000
  d_model: 512
  n_heads: 8
  n_kv_heads: 2         # GQA: 8 Q-голов / 2 K/V-группы = по 4 Q на группу
  n_layers: 8
  d_ff: 2048
  max_seq_len: 512
  dropout: 0.15

training:
  batch_size: 32
  learning_rate: 3e-4
  weight_decay: 0.05
  max_epochs: 10
  warmup_steps: 2000
  max_grad_norm: 1.0
  checkpoint_dir: /content/drive/MyDrive/CommonCrawl/checkpoints_lab4/
  log_dir: /content/drive/MyDrive/CommonCrawl/logs_lab4/

Overwriting /content/MNNA-2026/configs/gpt_config.yaml


In [25]:
print("=" * 60)
print("СОДЕРЖИМОЕ attention.py (первые 60 строк)")
print("=" * 60)
!head -60 /content/MNNA-2026/src/models/attention.py

print("\n" + "=" * 60)
print("СОДЕРЖИМОЕ lightning_module.py (первые 40 строк)")
print("=" * 60)
!head -40 /content/MNNA-2026/src/models/lightning_module.py

print("\n" + "=" * 60)
print("СОДЕРЖИМОЕ gpt_config.yaml")
print("=" * 60)
!cat /content/MNNA-2026/configs/gpt_config.yaml

СОДЕРЖИМОЕ attention.py (первые 60 строк)
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):
    """
    Многоголовое внимание с поддержкой:
      - packed batching (seq_ids),
      - GQA (n_kv_heads <= n_heads),
      - KV-кэш для инференса (past_kv).

    forward возвращает (out, new_kv), где new_kv = (k, v) — обновлённый кэш.
    Если past_kv=None — это первый вызов, кэш создаётся с нуля.
    """

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        if n_kv_heads is None:
            n_kv_heads = n_heads

        assert n_heads % n_kv_heads == 0, \
            f"n_heads ({n_heads}) must be divisible by n_kv_heads ({n_kv_heads})"

        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep 

In [26]:
import sys

# Удаляем все модули проекта из кэша
removed = []
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src') or mod_name == 'src':
        del sys.modules[mod_name]
        removed.append(mod_name)

print("Удалено из кэша:", removed)
print("\nТеперь заново импортируем...")

# ВАЖНО: добавляем путь ДО импорта
if '/content/MNNA-2026' not in sys.path:
    sys.path.insert(0, '/content/MNNA-2026')

# Проверяем, какой файл реально импортируется
import src.models.attention as attn_mod
print(f"\nФайл attention.py: {attn_mod.__file__}")

import inspect
sig = inspect.signature(attn_mod.MultiHeadAttention.__init__)
print(f"Сигнатура: {sig}")

# Проверяем, есть ли в исходнике n_kv_heads
with open(attn_mod.__file__) as f:
    src = f.read()
print(f"Есть 'n_kv_heads' в файле: {'n_kv_heads' in src}")
print(f"Есть 'bias=False' в файле: {'bias=False' in src}")
print(f"Есть 'bias=True' в файле: {'bias=True' in src}")

Удалено из кэша: ['src', 'src.models', 'src.models.positional_encoding', 'src.models.attention', 'src.models.ffn', 'src.models.transformer_layer', 'src.models.gpt', 'src.models.lightning_module']

Теперь заново импортируем...

Файл attention.py: /content/MNNA-2026/src/models/attention.py
Сигнатура: (self, d_model: int, n_heads: int, n_kv_heads: int = None, dropout: float = 0.1)
Есть 'n_kv_heads' в файле: True
Есть 'bias=False' в файле: True
Есть 'bias=True' в файле: False


In [27]:
from omegaconf import OmegaConf
from src.models.lightning_module import GPTLightningModule

cfg = OmegaConf.load('/content/MNNA-2026/configs/gpt_config.yaml')
print(f"cfg.model.n_kv_heads = {cfg.model.get('n_kv_heads', 'НЕТ')}")

model = GPTLightningModule(cfg)
attn = model.model.layers[0].attn

print(f"\nn_heads    = {attn.n_heads}")
print(f"n_kv_heads = {attn.n_kv_heads}")
print(f"W_q.weight = {attn.W_q.weight.shape}")
print(f"W_k.weight = {attn.W_k.weight.shape}")
print(f"W_v.weight = {attn.W_v.weight.shape}")
print(f"W_o.weight = {attn.W_o.weight.shape}")
print(f"W_k.bias   = {attn.W_k.bias}")

cfg.model.n_kv_heads = 2

n_heads    = 8
n_kv_heads = 2
W_q.weight = torch.Size([512, 512])
W_k.weight = torch.Size([128, 512])
W_v.weight = torch.Size([128, 512])
W_o.weight = torch.Size([512, 512])
W_k.bias   = None


In [28]:
import torch
from src.models.lightning_module import GPTLightningModule
from omegaconf import OmegaConf

cfg = OmegaConf.load('/content/MNNA-2026/configs/gpt_config.yaml')

checkpoint_path = "/content/drive/MyDrive/CommonCrawl/checkpoints_lab4/last.ckpt"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPTLightningModule.load_from_checkpoint(
    checkpoint_path,
    cfg=cfg,
    map_location=device,
    strict=False,   # <-- на всякий случай, чтобы игнорировать лишние ключи
)
model.eval()
model.to(device)
print("Модель загружена успешно!")

Модель загружена успешно!


In [29]:
import torch

@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    device = next(model.parameters()).device

    enc = tokenizer.encode(prompt)
    input_ids = torch.tensor([enc.ids], dtype=torch.long, device=device)
    seq_ids = torch.ones_like(input_ids)

    logits, past_kvs = model.model(
        input_ids, seq_ids, past_kvs=None, position_offset=0, use_cache=True
    )

    next_token_logits = logits[:, -1, :] / temperature
    if top_k > 0:
        v, _ = torch.topk(next_token_logits, top_k)
        next_token_logits[next_token_logits < v[..., -1, None]] = -float('Inf')
    probs = torch.softmax(next_token_logits, dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)

    generated_ids = torch.cat([input_ids, next_token], dim=1)

    for _ in range(max_new_tokens - 1):
        current_token = generated_ids[:, -1:]
        current_seq_id = torch.ones_like(current_token)
        pos_offset = generated_ids.shape[1] - 1
        logits, past_kvs = model.model(
            current_token, current_seq_id, past_kvs=past_kvs,
            position_offset=pos_offset, use_cache=True
        )
        next_token_logits = logits[:, -1, :] / temperature
        if top_k > 0:
            v, _ = torch.topk(next_token_logits, top_k)
            next_token_logits[next_token_logits < v[..., -1, None]] = -float('Inf')
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated_ids = torch.cat([generated_ids, next_token], dim=1)

    return tokenizer.decode(generated_ids[0].tolist())


# Проверим на нескольких промптах
prompts = [
    "The history of the world is",
    "In the beginning",
    "Albert Einstein was",
    "The city of London",
]

for p in prompts:
    print("=" * 70)
    print(f"Промпт: {p}")
    print("-" * 70)
    generated = generate_text(model, tokenizer, p, max_new_tokens=80, temperature=0.8, top_k=50)
    print(generated)
    print()

Промпт: The history of the world is
----------------------------------------------------------------------
The history of the world is the English - ch urch es to the ch urch of Church , which is one of the re main ing of the modern and ch urch and the 18 th cent ury 18 th cent ury , and the first ch urch was dated 5 , 000 , and the 18 th cent ury . The ch urch is the first of the ch urch ch urch . The Church of B ish op

Промпт: In the beginning
----------------------------------------------------------------------
In the beg inning of the third season in a season , he made his first appear ance with the second . He made a the first team to win the first time in the second inn ings to win the third in the nin th inning . He was the first inn ings for the final . He also the first time since his third appear ance , was nam ed the sem if in als . He was nam ed the final

Промпт: Albert Einstein was
----------------------------------------------------------------------
Al bert E inst ein